# VLM Architecture Study

## 1. Load the VLM

In [1]:
from unsloth import FastVisionModel
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/media/cosmic-muffin/wd-black/git/envs/prism/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

==((====))==  Unsloth 2026.2.1: Fast Qwen3_Vl patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 5050 Laptop GPU. Num GPUs = 1. Max memory: 7.525 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [3]:
print(f"Model Type: {type(model).__name__}")
print(f"Tokenizer Type: {type(tokenizer).__name__}")

NameError: name 'model' is not defined

## 2. Architecture Overview

**VLM has 3 distinct component**
 - **Vision Encoder**
 - **Project Layer**
 - **LLM Backbone**

### 2.1 ViT
 - **converts image into a sequence of "Visual tokens" each representing a patch of the image.**
 - **ViT are frozen while fine-tuning**

In [ ]:
vision_model = model.model.visual if hasattr(model.model, "visual") else model.visual_model
print(f"Vision encoder type: {type(vision_model).__name__}")
print("Vision encoder structure (top-level):")
for name, child in vision_model.named_children():
    param_count = sum(p.numel() for p in child.parameters())
    print(f"\t{name:25s} | {param_count/1e6:>6.1f}M Params | {type(child).__name__}")

In [ ]:
# The patch embedding dimensions
# patch size, number of visual tokens per image, and embedding dimensions
print("Search for patch/conv embedding layers...")
for name, module in vision_model.named_modules():
    if 'path' in name.lower() or 'embed' in name.lower() or isinstance(module, torch.nn.Conv2d):
        if hasattr(module, "weight"):
            print(f"\t{name}: {module}")
            if isinstance(module, torch.nn.Conv2d):
                print(f"\t -> Patch size: {module.kernel_size}, Stride: {module.stride}")
                print(f"\t -> Input channels: {module.in_channels}, Output dimensions: {module.out_channels}")

### 2.2 Projection Layer
 - **bridges the gap between the visual tokens and the LLM**
 - **usually a small MLP layer**
 - **maps the visual token dimension to the LLM embedding dimension**
 - **can be frozen or fine-tuned**

`Linear(d_vision, d_llm)` or `Linear → GELU → Linear`

In [ ]:
print("Search for projection layers...")
for name, module in model.named_modules():
    if any(kw in name.lower() for kw in ["merger", "project", "connector", "multi_modal"]):
        if hasattr(module, "weight") or list(module.children()):
            param_count = sum(p.numel() for p in module.parameters())
            if param_count > 0:
                print(f"\t{name}")
                print(f"\tType: {type(module).__name__}")
                print(f"\tParams: {param_count/1e6:.1f}M")

                # Print shape if it's a linear layer
                if hasattr(module, "weight"):
                    print(f"\tWeight shape: {module.weight.shape}")
                    print(f"\t -> Maps d_vision={module.weight.shape[1]} -> d_llm={module.weight.shape[0]}", end="\n\n")

### 2.3 LLM Backbone

In [ ]:
llm_layers = [name for name, _ in model.named_modules() if 'layers.' in name.endswith('.self_attn')]
print(f"LLM backbone has {len(llm_layers)} transformer layers")
print("\nFirst 3 layer names (for reference):")
for layer in llm_layers[:3]:
    print(f"\t{layer}")
    

# 3. Parameter Counts

In [ ]:
def count_params(module):
    return sum(p.numel() for p in module.parameters())

total = count_params(model)

vision_params = 0
llm_params = 0
other_params = 0

for name, param in model.named_parameters():
    if 'visual' in name:
        vision_params += param.numel()
    else:
        llm_params += param.numel()

print(f"{'Component':<25} {'Params':>12} {'% of total':>10}")
print("="*50)
print(f"{'Vision Encoder':<25} {vision_params/1e6:>10.1f}M {vision_params/total*100:>9.1f}%")
print(f"{'LLM + Projection':<25} {llm_params/1e6:>10.1f}M {llm_params/total*100:>9.1f}%")
print("="*50)
print(f"{'Total':<25} {total/1e6:>10.1f}M {100:>9.1f}%")
print(f"\nWith LoRA (rank=16, ~0.5-1% of LLM), you'd train ~{llm_params*0.007/1e6:.1f}M parameters")

SyntaxError: f-string: valid expression required before '}' (2111122431.py, line 18)

# 4. Applying LoRA and architecture overview

In [ ]:
model_v1 = FastVisionModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_rslora=False,
    loftq_config=None,
)

In [ ]:
trainable_v1 = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_v1 = sum(p.numel() for p in model.parameters() if not p.requires_grad)
total_v1 = trainable_v1 + frozen_v1

print(f"Trainable: {trainable_v1/1e6:.1f}M ({trainable_v1/total_v1*100:.2f}%)")
print(f"Frozen:    {frozen_v1/1e6:.1f}M ({frozen_v1/total_v1*100:.2f}%)")
print(f"Total:     {total_v1/1e6:.1f}M")

In [ ]:
model_v2 = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=47,
    use_rslora=False,
    loftq_config=None,
)

In [ ]:
# displaying exactly which layers have LoRA adapters
print("Layers with LoRA adapters:\n")
lora_layers = []
for name, param in model.named_parameters():
    if param.requires_grad and 'lora' in name:
        lora_layers.append(name)

# Displaying first few and last few layers
for layer in lora_layers[:6]:
    print("\t ✓ {layer}")
print(f"\t.... ({len(lora_layers) - 12} more layers) ....")
for layer in lora_layers[-6:]:
    print("\t ✓ {layer}")

print(f"\nTotal LoRA Layers: {len(lora_layers)}")

vision_trainable = sum(p.numel() for n, p in model.named_parameters() if 'visual' in n and p.requires_grad)
print(f"\nVision encoder trainable parameters: {vision_trainable} (should be 0) because vision layer is frozen")

In [ ]:
trainable_v2 = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_v2 = sum(p.numel() for p in model.parameters() if not p.requires_grad)
total_v2 = trainable_v2 + frozen_v2

print(f"Trainable: {trainable_v2/1e6:.1f}M ({trainable_v2/total_v2*100:.2f}%)")
print(f"Frozen:    {frozen_v2/1e6:.1f}M ({frozen_v2/total_v2*100:.2f}%)")
print(f"Total:     {total_v2/1e6:.1f}M")

In [ ]:
# displaying exactly which layers have LoRA adapters
print("Layers with LoRA adapters:\n")
lora_layers = []
for name, param in model.named_parameters():
    if param.requires_grad and 'lora' in name:
        lora_layers.append(name)

# Displaying first few and last few layers
for layer in lora_layers[:6]:
    print("\t ✓ {layer}")
print(f"\t.... ({len(lora_layers) - 12} more layers) ....")
for layer in lora_layers[-6:]:
    print("\t ✓ {layer}")

print(f"\nTotal LoRA Layers: {len(lora_layers)}")

vision_trainable = sum(p.numel() for n, p in model.named_parameters() if 'visual' in n and p.requires_grad)
print(f"\nVision encoder trainable parameters: {vision_trainable} (should be 0) because vision layer is frozen")

# 5. Inference Mode - Testing

In [ ]:
from PIL import Image
import requests
from io import BytesIO

url = "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Camponotus_flavomarginatus_ant.jpg/320px-Camponotus_flavomarginatus_ant.jpg"
response = requests.get(url)
image = Image.open(BytesIO(response.content)).convert("RGB")

print(f"Image size: {image.size} (WxH)")
print(f"Image mode: {image.mode}")
image

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "Describe what you see in this image in one sentence."},
        ],
    }
]

input_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print("Formatted input (first 500 chars):")
print(input_text[:500])
print("\n...")
print(input_text[-200:])

In [ ]:
# inferencing
FastVisionModel.for_inference(model_v1)

from qwen_vl_utils import process_vision_info

image_inputs, video_inputs = process_vision_info(messages)

inputs = tokenizer(
    text=[input_text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
).to(model_v1.device)

print(f"Input token count: {inputs['input_ids'].shape[1]}")
print("\t (This includes both visual tokens from the image and text tokens from the prompt)")

with torch.no_grad():
    output_ids = model_v1.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.7,
        do_sample=True,
    )

generated = tokenizer.decode(
    output_ids[0][inputs['input_ids'].shape[1]:],
    skip_special_tokens=True,
)

print(f"\nModel response: {generated}")

# 6. Trace Token Dimensions

In [ ]:
input_ids = inputs['input_ids'][0]

# Special vision tokens in the input
# Qwen3-vl uses <|vision_start|>, <|image_pad|>, <|vision_end|> tokens
special_tokens = {
    "vision_start": tokenizer.convert_tokens_to_ids("<|vision_start|>"),
    "vision_end": tokenizer.convert_tokens_to_ids("<|vision_end|>"),
    "image_pad": tokenizer.convert_tokens_to_ids("<|image_pad|>"),
}

print("Special vision token IDs:")
for name, tid in special_tokens.items():
    if tid is not None:
        count = (input_ids == tid).sum().item()
        print(f"\t{name}: ID={tid}, appears {count} times")

# Count visual vs text tokens
if special_tokens['image_pad'] is not None:
    n_visual = (input_ids == special_tokens['image_pad']).sum().item()
    n_total = len(input_ids)
    n_text = n_total - n_visual
    print("\nToken breakdown:")
    print(f"\tVisual tokens: {n_visual}")
    print(f"\tText tokens: {n_text}")
    print(f"\tTotal: {n_total}")
    print(f"\tVisual ratio: {n_visual/n_total*100:.1f}%")
    print(f"\n-> One image generates ~{n_visual} tokens")

In [ ]:
# check the LLM's hidden dimension - this is what projection maps TO
config = model.config
print("Model configuration (key dimensions):")
print(f"\tLLM hidden size (d_llm): {config.hidden_size}")
print(f"\tLLM num layers: {config.num_hidden_layers}")
print(f"\tLLM attention heads: {config.num_attention_heads}")
print(f"\tLLM vocab size: {config.vocab_size}")

# Vision config if available
if hasattr(config, "vision_config"):
    vc = config.vision_config
    print(f"\n\tVision hidden size: {getattr(vc, 'hidden_size', 'N/A')}")
    print(f"\t\tVision num layers: {getattr(vc, 'num_hidden_layers', 'N/A')}")
    print(f"\t\tVision patch size: {getattr(vc, 'patch_size', 'N/A')}")